# SFT-CL Phase B — CIFAR-100 on Colab

This notebook freezes ViT-B/16 and extracts features at most once. It then selects SFT hyperparameters only from cached **training** features, locks them, and evaluates global task-free classifiers on test features. It does **not** fine-tune ViT.

Use a GPU runtime. The disk cache is experiment infrastructure, not learner state. `cached_soho_replay` is explicitly non-exemplar-free because it retains per-example feature history.

In [ ]:
# === Edit this cell only ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/sft-cl-exemplar-free'  # push this branch before running
CHECKPOINT_SOURCE = 'huggingface'  # 'huggingface' or 'google_drive'
DRIVE_CHECKPOINT_PATH = '/content/drive/MyDrive/T-SOHO/model.safetensors'
WORK_DIR = '/content/SOHO-CL'
# Reuse your previous T-SOHO cache if its metadata has the same ViT checkpoint/preprocessing.
CACHE_DIR = '/content/tsoho_cifar100_cache'
OUTPUT_DIR = '/content/sft_cl_phase_b_outputs'
SEED = 1993
NUM_TASKS = 10
BATCH_SIZE = 128
VALIDATION_FRACTION = 0.10
SOFT_RIDGE_LAMBDAS = '0.01,0.1,1.0'
KAPPAS = '0.01,0.1,1.0'
DELTAS = '0.01,0.1,0.5'
HARD_RANKS = '32,64,128,256'
HARD_RIDGE_LAMBDAS = '0.01,0.1,1.0'
# Optional legacy controls: values must come from a separately locked baseline protocol.
RUN_TSOHO_V1_CONTROL = False
TSOHO_V1_RANK = 98
TSOHO_V1_RIDGE_LAMBDA = 0.01
RUN_LEGACY_CACHE_CONTROLS = False
FLY_LOCKED_RIDGE_LAMBDA = None
SOHO_LOCKED_RIDGE_LAMBDA = None
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'


In [ ]:
import os, shutil, subprocess, torch
assert torch.cuda.is_available(), 'In Colab select Runtime > Change runtime type > T4 GPU.'
!nvidia-smi
# The notebook may currently be inside WORK_DIR from an earlier run. Leave
# it before deleting, otherwise the shell loses its current directory.
%cd /content
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
%cd {WORK_DIR}
!pip -q install -r requirements-kaggle.txt kagglehub huggingface_hub
if CHECKPOINT_SOURCE == 'google_drive':
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
elif CHECKPOINT_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
else:
    raise ValueError("CHECKPOINT_SOURCE must be 'huggingface' or 'google_drive'")
print('checkpoint:', CHECKPOINT_PATH)
!git log -1 --oneline


In [ ]:
# Public CIFAR-100 only. Do not download CUB or ImageNet-R in this notebook.
from pathlib import Path
import kagglehub
downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
candidates = [downloaded, *downloaded.rglob('cifar-100')]
cifar_dir = next(p for p in candidates if (p / 'train').is_file() and (p / 'test').is_file() and (p / 'meta').is_file())
CIFAR_ROOT = str(cifar_dir)
print('CIFAR-100:', CIFAR_ROOT)
assert Path(CHECKPOINT_PATH).is_file(), CHECKPOINT_PATH


In [ ]:
# Check checkpoint identity, model load, and all relevant unit tests.
!python tools/checkpoint_preflight.py --root {CIFAR_ROOT} --checkpoint {CHECKPOINT_PATH} --checkpoint-size {CHECKPOINT_SIZE} --checkpoint-sha256 {CHECKPOINT_SHA256} --seed {SEED} --batch-size {BATCH_SIZE}
!python -m pytest -q tests/test_sft_cl_math.py tests/test_cached_replay_baselines.py tests/test_experiment_runner.py tests/test_tsoho_learner.py tests/test_tsoho_math.py tests/test_backbone_checkpoint.py


In [ ]:
# Extract frozen ViT features once. Existing validated cache is reused.
from pathlib import Path
if not Path(CACHE_DIR, 'metadata.json').is_file():
    cmd = f'''python tools/experiment_runner.py --extract-features-only --root {CIFAR_ROOT} --backbone-checkpoint {CHECKPOINT_PATH} --backbone-checkpoint-size {CHECKPOINT_SIZE} --backbone-checkpoint-sha256 {CHECKPOINT_SHA256} --feature-cache-dir {CACHE_DIR} --output-dir {OUTPUT_DIR}/cache_extract --dataset CIFAR-100 --model-name vit_base_patch16_224 --data-augmentation vit --seed {SEED} --num-classes 100 --num-tasks {NUM_TASKS} --device cuda --batch-size {BATCH_SIZE} --num-workers 2'''
    assert os.system(cmd) == 0, cmd
else:
    print('Using existing cache:', CACHE_DIR)


In [ ]:
# Selection A: proposed confusion-soft Fisher. This opens TRAIN cache only.
import json
soft_selection_path = f'{OUTPUT_DIR}/selection/confusion_soft.json'
cmd = f'''python tools/experiment_runner.py --select-config --config configs/sft_cl_cifar100_template.json --feature-cache-dir {CACHE_DIR} --output-dir {OUTPUT_DIR}/selection --selection-output {soft_selection_path} --dataset CIFAR-100 --model-name vit_base_patch16_224 --num-classes 100 --num-tasks {NUM_TASKS} --seed {SEED} --device cuda --search-methods confusion_fisher_soft --search-lambdas {SOFT_RIDGE_LAMBDAS} --search-kappas {KAPPAS} --search-deltas {DELTAS} --validation-fraction {VALIDATION_FRACTION}'''
assert os.system(cmd) == 0, cmd
soft_selected = json.load(open(soft_selection_path))['best']
print('LOCKED proposed SFT config (train-only):', soft_selected)


In [ ]:
# Selection B: hard Fisher only. It has rank; this also opens TRAIN cache only.
hard_selection_path = f'{OUTPUT_DIR}/selection/fisher_hard.json'
cmd = f'''python tools/experiment_runner.py --select-config --feature-cache-dir {CACHE_DIR} --output-dir {OUTPUT_DIR}/selection --selection-output {hard_selection_path} --dataset CIFAR-100 --model-name vit_base_patch16_224 --num-classes 100 --num-tasks {NUM_TASKS} --seed {SEED} --device cuda --search-methods fisher_hard --search-ranks {HARD_RANKS} --search-lambdas {HARD_RIDGE_LAMBDAS} --validation-fraction {VALIDATION_FRACTION}'''
assert os.system(cmd) == 0, cmd
hard_selected = json.load(open(hard_selection_path))['best']
print('LOCKED hard-Fisher config (train-only):', hard_selected)


In [ ]:
# Final paired test evaluation. Do not modify soft_selected/hard_selected after this point.
base = f'--feature-cache-dir {CACHE_DIR} --dataset CIFAR-100 --model-name vit_base_patch16_224 --num-classes 100 --num-tasks {NUM_TASKS} --seed {SEED} --device cuda --resume'
final_runs = [
    ('sft_raw_ridge', f'--ridge-lambda {soft_selected["ridge_lambda"]}'),
    ('fisher_soft', f'--ridge-lambda {soft_selected["ridge_lambda"]} --fisher-kappa {soft_selected["fisher_kappa"]} --fisher-delta {soft_selected["fisher_delta"]}'),
    ('confusion_fisher_soft', f'--ridge-lambda {soft_selected["ridge_lambda"]} --fisher-kappa {soft_selected["fisher_kappa"]} --fisher-delta {soft_selected["fisher_delta"]}'),
    ('shuffled_confusion_fisher_soft', f'--ridge-lambda {soft_selected["ridge_lambda"]} --fisher-kappa {soft_selected["fisher_kappa"]} --fisher-delta {soft_selected["fisher_delta"]}'),
    ('fisher_hard', f'--rank {hard_selected["rank"]} --ridge-lambda {hard_selected["ridge_lambda"]}'),
]
for method, locked_args in final_runs:
    run_dir = f'{OUTPUT_DIR}/final_{method}'
    cmd = f'python tools/experiment_runner.py --method {method} {locked_args} {base} --output-dir {run_dir}'
    assert os.system(cmd) == 0, cmd


In [ ]:
# Optional controls. These are not selected/tuned here.
if RUN_TSOHO_V1_CONTROL:
    cmd = f'python tools/experiment_runner.py --method spectral_confusion_code --rank {TSOHO_V1_RANK} --ridge-lambda {TSOHO_V1_RIDGE_LAMBDA} {base} --output-dir {OUTPUT_DIR}/final_tsoho_v1'
    assert os.system(cmd) == 0, cmd

if RUN_LEGACY_CACHE_CONTROLS:
    assert FLY_LOCKED_RIDGE_LAMBDA is not None and SOHO_LOCKED_RIDGE_LAMBDA is not None, 'Use ridge values locked by the legacy baseline protocol.'
    controls = [('cached_flycl', FLY_LOCKED_RIDGE_LAMBDA), ('cached_soho_replay', SOHO_LOCKED_RIDGE_LAMBDA)]
    for method, ridge in controls:
        cmd = f'python tools/experiment_runner.py --method {method} --ridge-lambda {ridge} {base} --output-dir {OUTPUT_DIR}/final_{method}'
        assert os.system(cmd) == 0, cmd
    print('Reminder: cached_soho_replay is a replay baseline; metrics.json must say exemplar_free=False.')


In [ ]:
# Aggregate and download evidence. Inspect the table before making any research claim.
import glob, pandas as pd
rows = []
for path in glob.glob(f'{OUTPUT_DIR}/final_*/metrics.json'):
    result = json.load(open(path))
    result['method'] = Path(path).parent.name.replace('final_', '')
    rows.append(result)
table = pd.DataFrame(rows).sort_values('method')
display(table)
!zip -r /content/sft_cl_phase_b_artifacts.zip {OUTPUT_DIR}
from google.colab import files
files.download('/content/sft_cl_phase_b_artifacts.zip')
